# Baseline: YOLO Food Detection Pipeline

COCO annotations → YOLO dataset + `dataset.yaml` → train → evaluate → detect → test on your own photos → save `.pt` + `.onnx`.

**Runs locally / on Colab.** Downloads happen only in the install cell and when Ultralytics fetches pretrained weights. The saved model runs offline afterwards (ONNX + onnxruntime).

**Licensing:** Ultralytics is AGPL-3.0, so keep your repo open source and credit the dataset (e.g. MyFoodRepo / Food Recognition Challenge) in `THIRD_PARTY.md`. Verify the dataset license for the exact version you download.

Sections: 0 Setup · 1 Config · 2 Inspect COCO · 3 COCO→YOLO · 4 Check labels · 5 Train · 6 Evaluate · 7 Detection function · 8 Test on team photos · 9 Save model

## 0. Setup

In [1]:
# Pin versions you have tested. Prefer releases at least two weeks old.
# If YOLO26 weights are not found, update ultralytics or use the YOLO11n fallback set in the config.
%pip install -q ultralytics onnx onnxruntime pyyaml pandas matplotlib pillow rapidfuzz

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import json, os, random, shutil, sys, time, csv, yaml
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
import ultralytics
print("ultralytics", ultralytics.__version__)

# Colab only: uncomment to keep data and runs on Drive
# from google.colab import drive; drive.mount('/content/drive')

OSError: [WinError 4551] An Application Control policy has blocked this file. Error loading "c:\Users\Chinmayee\miniconda3\envs\myenv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 1. Config

Edit this cell only. Paths follow the repo layout from the Kiro prompt pack (`data/food/...`, `ml/food/runs`).

- If you have **separate train and val COCO files**, set both. If you only have a train file, set `val_json = None` and a random split by image is used.
- No GPU? Lower `epochs`, set `fraction` below 1.0 (uses a subset of training images), and keep `imgsz=640` or drop to 512.

In [ ]:
CFG = dict(
    # ---- input data (COCO format) ----
    train_json   = "data/food/train/annotations.json",
    train_images = "data/food/train/images",
    val_json     = "data/food/val/annotations.json",   # set to None for a random split
    val_images   = "data/food/val/images",             # set to None for a random split
    val_frac     = 0.1,                                # used only when val_json is None

    # ---- conversion ----
    out          = "data/food_yolo",       # YOLO dataset root (images/, labels/, dataset.yaml)
    top_n        = 60,                     # keep the N most frequent classes (50-80 recommended)
    min_box_px   = 12,                     # drop boxes smaller than this
    task         = "detect",               # "detect" or "segment" (segment needs a -seg model)
    symlink      = False,                  # True saves disk on Linux/Mac; keep False on Windows/Colab Drive

    # ---- training ----
    model        = "yolo26n.pt",           # preferred
    fallback     = "yolo11n.pt",           # used if the preferred weights fail to load
    imgsz        = 640,
    epochs       = 50,
    batch        = 16,
    fraction     = 1.0,                    # fraction of training images (0.1-1.0) for quick runs
    patience     = 15,
    workers      = 2,
    seed         = 42,
    device       = None,                   # None = auto (GPU if available); or 0 / "cpu"
    project      = "ml/food/runs",
    name         = "food_yolo_v1",

    # ---- inference / app behaviour ----
    conf         = 0.25,                   # detection confidence threshold
    unsure_below = 0.40,                   # if the best score is below this, mark result unsure
    iou          = 0.60,

    # ---- optional: your own photos for the end-to-end test ----
    team_photos_dir = "data/team_photos",  # contains images + labels.csv (image,item,fdc_id)

    # ---- saved model ----
    save_dir     = "backend/food/model",
)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"])

## 2. Inspect the COCO input

In [ ]:
def load_coco(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for key in ("images", "annotations", "categories"):
        if key not in data:
            raise ValueError(f"{path} is missing '{key}' (not COCO format?)")
    return data

train_coco = load_coco(CFG["train_json"])
id2name = {c["id"]: c["name"] for c in train_coco["categories"]}
counts = Counter(id2name.get(a["category_id"]) for a in train_coco["annotations"] if not a.get("iscrowd", 0))

print(f"images: {len(train_coco['images'])} | annotations: {len(train_coco['annotations'])} | classes: {len(id2name)}")
print("\nTop 15 classes by annotation count:")
for n, c in counts.most_common(15):
    print(f"  {c:6d}  {n}")

# class frequency (long tail check)
vals = [c for _, c in counts.most_common()]
plt.figure(figsize=(9, 3)); plt.plot(vals); plt.axvline(CFG["top_n"], color="r", ls="--", label=f"top_n={CFG['top_n']}")
plt.yscale("log"); plt.xlabel("class rank"); plt.ylabel("annotations"); plt.legend(); plt.title("Class frequency"); plt.show()

## 3. Convert COCO → YOLO and write `dataset.yaml`

Same logic as `ml/food/coco_to_yolo.py`, inlined so the notebook is self-contained. It keeps the top-N classes (matched **by name** across train and val files), drops crowd annotations and tiny boxes, clips boxes to the image, and writes `dataset.yaml`, `classes.txt` and `conversion_report.json`.

In [ ]:
def select_class_names(coco, top_n):
    idn = {c["id"]: c["name"] for c in coco["categories"]}
    cnt = Counter()
    for a in coco["annotations"]:
        if a.get("iscrowd", 0): continue
        n = idn.get(a["category_id"])
        if n is not None: cnt[n] += 1
    return [n for n, _ in cnt.most_common(top_n)], cnt

def cat_map_for(coco, name_to_idx):
    return {c["id"]: name_to_idx[c["name"]] for c in coco["categories"] if c["name"] in name_to_idx}

def bbox_line(ann, w, h, idx, min_px):
    x, y, bw, bh = ann["bbox"]
    if bw < min_px or bh < min_px: return None
    x0, y0 = max(0.0, x), max(0.0, y)
    x1, y1 = min(float(w), x + bw), min(float(h), y + bh)
    if x1 <= x0 or y1 <= y0: return None
    cx, cy = (x0 + x1) / 2 / w, (y0 + y1) / 2 / h
    return f"{idx} {cx:.6f} {cy:.6f} {(x1-x0)/w:.6f} {(y1-y0)/h:.6f}"

def seg_line(ann, w, h, idx, min_px):
    seg = ann.get("segmentation")
    if not isinstance(seg, list) or not seg: return None      # RLE or empty: skip
    if ann["bbox"][2] < min_px or ann["bbox"][3] < min_px: return None
    poly = max(seg, key=len)
    if len(poly) < 6: return None
    co = []
    for i in range(0, len(poly) - 1, 2):
        co += [f"{min(max(poly[i]/w, 0.0), 1.0):.6f}", f"{min(max(poly[i+1]/h, 0.0), 1.0):.6f}"]
    return f"{idx} " + " ".join(co)

def place_image(src, dst, symlink):
    if dst.exists() or dst.is_symlink(): return
    if symlink: os.symlink(src.resolve(), dst)
    else: shutil.copy2(src, dst)

def convert_split(coco, img_dir, out_root, split, cat_to_idx, task, min_px, symlink, image_ids=None):
    img_out = out_root / "images" / split; lbl_out = out_root / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True); lbl_out.mkdir(parents=True, exist_ok=True)
    by_img = {}
    for a in coco["annotations"]:
        if a.get("iscrowd", 0) or a["category_id"] not in cat_to_idx: continue
        by_img.setdefault(a["image_id"], []).append(a)
    make_line = seg_line if task == "segment" else bbox_line
    stats, per_class = Counter(), Counter()
    for im in coco["images"]:
        if image_ids is not None and im["id"] not in image_ids: continue
        anns = by_img.get(im["id"], [])
        if not anns: stats["images_skipped_no_labels"] += 1; continue
        src = Path(img_dir) / im["file_name"]
        if not src.exists(): stats["images_missing"] += 1; continue
        w, h = im["width"], im["height"]
        lines = []
        for a in anns:
            idx = cat_to_idx[a["category_id"]]
            ln = make_line(a, w, h, idx, min_px)
            if ln is None: stats["annotations_dropped"] += 1; continue
            lines.append(ln); per_class[idx] += 1
        if not lines: stats["images_skipped_no_valid_boxes"] += 1; continue
        stem = str(im["id"])
        place_image(src, img_out / f"{stem}{src.suffix.lower()}", symlink)
        (lbl_out / f"{stem}.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
        stats["images_written"] += 1; stats["annotations_written"] += len(lines)
    return stats, per_class

def write_yaml(out_root, names):
    data = {"path": out_root.resolve().as_posix(), "train": "images/train", "val": "images/val",
            "names": {i: n for i, n in enumerate(names)}}
    (out_root / "dataset.yaml").write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")
    (out_root / "classes.txt").write_text("\n".join(names) + "\n", encoding="utf-8")

def coco_to_yolo(cfg):
    out_root = Path(cfg["out"])
    if out_root.exists(): shutil.rmtree(out_root)          # clean rebuild so stale labels cannot leak in
    train = load_coco(cfg["train_json"])
    names, cnt = select_class_names(train, cfg["top_n"])
    if not names: raise SystemExit("no usable annotations found")
    n2i = {n: i for i, n in enumerate(names)}
    report = {"classes": names, "annotation_counts_top": cnt.most_common(cfg["top_n"])}
    if cfg["val_json"]:
        val = load_coco(cfg["val_json"])
        s_tr, pc_tr = convert_split(train, cfg["train_images"], out_root, "train", cat_map_for(train, n2i), cfg["task"], cfg["min_box_px"], cfg["symlink"])
        s_va, pc_va = convert_split(val, cfg["val_images"], out_root, "val", cat_map_for(val, n2i), cfg["task"], cfg["min_box_px"], cfg["symlink"])
    else:
        cmap = cat_map_for(train, n2i)
        labeled = sorted({a["image_id"] for a in train["annotations"] if not a.get("iscrowd", 0) and a["category_id"] in cmap})
        rng = random.Random(cfg["seed"]); rng.shuffle(labeled)
        n_val = max(1, int(len(labeled) * cfg["val_frac"])) if len(labeled) > 1 else 0
        val_ids, train_ids = set(labeled[:n_val]), set(labeled[n_val:])
        s_tr, pc_tr = convert_split(train, cfg["train_images"], out_root, "train", cmap, cfg["task"], cfg["min_box_px"], cfg["symlink"], train_ids)
        s_va, pc_va = convert_split(train, cfg["train_images"], out_root, "val", cmap, cfg["task"], cfg["min_box_px"], cfg["symlink"], val_ids)
    write_yaml(out_root, names)
    report.update(train=dict(s_tr), val=dict(s_va),
                  train_boxes_per_class={names[i]: c for i, c in sorted(pc_tr.items())},
                  val_boxes_per_class={names[i]: c for i, c in sorted(pc_va.items())})
    (out_root / "conversion_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    return names, report

names, report = coco_to_yolo(CFG)
DATA_YAML = Path(CFG["out"]) / "dataset.yaml"
print("classes kept:", len(names))
print("train:", report["train"]); print("val:  ", report["val"])
print("\n" + DATA_YAML.read_text()[:600] + "...")
if report["train"].get("images_missing", 0):
    print("\nWARNING: some images listed in the COCO file were not found. Check train_images/val_images paths.")

## 4. Sanity-check the converted labels (look at a few images)

In [ ]:
def show_labeled(split="train", n=6, seed=0):
    img_dir = Path(CFG["out"]) / "images" / split; lbl_dir = Path(CFG["out"]) / "labels" / split
    files = sorted(img_dir.glob("*.*")); random.Random(seed).shuffle(files)
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, f in zip(axes.ravel(), files[:n]):
        im = Image.open(f).convert("RGB"); W, H = im.size
        ax.imshow(im); ax.axis("off")
        lf = lbl_dir / f"{f.stem}.txt"
        for line in lf.read_text().splitlines():
            p = line.split(); cls = int(p[0])
            if CFG["task"] == "detect":
                cx, cy, w, h = map(float, p[1:5])
                x0, y0 = (cx - w/2) * W, (cy - h/2) * H
                ax.add_patch(plt.Rectangle((x0, y0), w * W, h * H, fill=False, ec="lime", lw=2))
                ax.text(x0, y0, names[cls], color="black", fontsize=8, bbox=dict(fc="lime", alpha=0.7, pad=1))
            else:
                pts = np.array(list(map(float, p[1:]))).reshape(-1, 2) * [W, H]
                ax.plot(*np.vstack([pts, pts[:1]]).T, color="lime", lw=2)
        ax.set_title(f.name, fontsize=8)
    plt.tight_layout(); plt.show()

show_labeled("train")

## 5. Train

Fine-tunes a pretrained small YOLO on your food classes. Falls back to `yolo11n.pt` if the preferred weights are unavailable in your installed Ultralytics version. Metrics, plots, and `weights/best.pt` land in `project/name`.

In [ ]:
from ultralytics import YOLO

def load_model(cfg):
    task_suffix = "-seg" if cfg["task"] == "segment" else ""
    for w in (cfg["model"], cfg["fallback"]):
        w = w.replace(".pt", f"{task_suffix}.pt") if task_suffix and task_suffix not in w else w
        try:
            m = YOLO(w); print("loaded weights:", w); return m, w
        except Exception as e:
            print(f"could not load {w}: {type(e).__name__}: {str(e)[:120]}")
    raise RuntimeError("no usable pretrained weights. Update ultralytics or check your network.")

model, used_weights = load_model(CFG)

t0 = time.time()
results = model.train(
    data=str(DATA_YAML), epochs=CFG["epochs"], imgsz=CFG["imgsz"], batch=CFG["batch"],
    fraction=CFG["fraction"], patience=CFG["patience"], workers=CFG["workers"],
    seed=CFG["seed"], device=CFG["device"], project=CFG["project"], name=CFG["name"],
    exist_ok=True, plots=True,
)
print(f"training done in {(time.time()-t0)/60:.1f} min")

RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
print("best weights:", BEST, "| exists:", BEST.exists())

In [ ]:
# Training curves
res = pd.read_csv(RUN_DIR / "results.csv"); res.columns = [c.strip() for c in res.columns]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for c in [c for c in res.columns if "loss" in c and c.startswith("train")]: ax[0].plot(res["epoch"], res[c], label=c)
ax[0].set_title("train loss"); ax[0].legend(fontsize=7)
for c in [c for c in res.columns if "loss" in c and c.startswith("val")]: ax[1].plot(res["epoch"], res[c], label=c)
ax[1].set_title("val loss"); ax[1].legend(fontsize=7)
for c in [c for c in res.columns if "mAP" in c]: ax[2].plot(res["epoch"], res[c], label=c)
ax[2].set_title("mAP"); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 6. Evaluate on the validation split

In [ ]:
best = YOLO(str(BEST))
metrics = best.val(data=str(DATA_YAML), split="val", imgsz=CFG["imgsz"], conf=0.001, iou=CFG["iou"], plots=True, verbose=False)
box = metrics.box
print(f"mAP50     : {box.map50:.3f}")
print(f"mAP50-95  : {box.map:.3f}")
print(f"precision : {box.mp:.3f} | recall: {box.mr:.3f}")

per_class = pd.DataFrame({"class": [names[i] for i in box.ap_class_index],
                          "AP50-95": box.maps[box.ap_class_index]}).sort_values("AP50-95")
print("\nWeakest 10 classes (candidates to merge, drop, or add photos for):")
print(per_class.head(10).to_string(index=False))
print("\nStrongest 5:"); print(per_class.tail(5).to_string(index=False))

## 7. Detection function (matches the app's `DetectionResult` contract)

Returns normalized `[x, y, w, h]` boxes (0–1), a score and a label per box, and `unsure=True` when nothing is detected or the best score is below `unsure_below`. A plain YOLO detector gives one label per box. The **top-3 labels** the app shows can come from the optional OpenCLIP second stage, or you can add them later by running a class-probability pass. Portions are chosen by the user, never from box size.

In [ ]:
def detect(model, image_path, conf=None, iou=None, imgsz=None, unsure_below=None):
    conf = CFG["conf"] if conf is None else conf
    iou = CFG["iou"] if iou is None else iou
    imgsz = CFG["imgsz"] if imgsz is None else imgsz
    unsure_below = CFG["unsure_below"] if unsure_below is None else unsure_below
    r = model.predict(str(image_path), conf=conf, iou=iou, imgsz=imgsz, verbose=False)[0]
    H, W = r.orig_shape
    dets = []
    for b in r.boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        cls = int(b.cls[0]); score = float(b.conf[0])
        dets.append({"box": [round(x1/W, 4), round(y1/H, 4), round((x2-x1)/W, 4), round((y2-y1)/H, 4)],
                     "score": round(score, 4),
                     "labels": [{"label": r.names[cls], "score": round(score, 4)}]})
    dets.sort(key=lambda d: -d["score"])
    unsure = (len(dets) == 0) or (dets[0]["score"] < unsure_below)
    return {"image_id": Path(image_path).name, "model_version": CFG["name"], "unsure": unsure, "detections": dets}

def show_detections(model, paths, cols=3):
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows)); axes = np.atleast_1d(axes).ravel()
    for ax, p in zip(axes, paths):
        res = detect(model, p); im = Image.open(p).convert("RGB"); W, H = im.size
        ax.imshow(im); ax.axis("off")
        for d in res["detections"]:
            x, y, w, h = d["box"]
            ax.add_patch(plt.Rectangle((x*W, y*H), w*W, h*H, fill=False, ec="red", lw=2))
            ax.text(x*W, y*H, f'{d["labels"][0]["label"]} {d["score"]:.2f}', color="white", fontsize=8, bbox=dict(fc="red", alpha=0.7, pad=1))
        ax.set_title(("UNSURE  " if res["unsure"] else "") + Path(p).name, fontsize=9)
    for ax in axes[len(paths):]: ax.axis("off")
    plt.tight_layout(); plt.show()

val_imgs = sorted((Path(CFG["out"]) / "images" / "val").glob("*.*"))
sample = random.Random(1).sample(val_imgs, min(6, len(val_imgs)))
show_detections(best, sample)
print(json.dumps(detect(best, sample[0]), indent=2)[:700])

## 8. Test on your own meal photos (end-to-end)

Put 100–150 photos in `data/team_photos/` with a `labels.csv` (`image,item,fdc_id`, one row per item on the plate). This measures what matters for the demo: **item recall**, **precision**, and a **user-correction rate** estimate. Photos are only read in memory here. The cell skips itself if the folder is missing.

In [ ]:
from rapidfuzz import fuzz

def norm(s): return " ".join(str(s).lower().replace("_", " ").replace(",", " ").split())
def same_item(gt, pred, thr=80): return norm(gt) == norm(pred) or fuzz.token_set_ratio(norm(gt), norm(pred)) >= thr

tp_dir = Path(CFG["team_photos_dir"]); lab_csv = tp_dir / "labels.csv"
if lab_csv.exists():
    gt = pd.read_csv(lab_csv)
    rows = []
    for img, grp in gt.groupby("image"):
        p = tp_dir / img
        if not p.exists(): continue
        res = detect(best, p)
        preds = [d["labels"][0]["label"] for d in res["detections"]]
        gt_items = grp["item"].tolist()
        hits = [any(same_item(g, pr) for pr in preds) for g in gt_items]
        prec_hits = [any(same_item(g, pr) for g in gt_items) for pr in preds]
        rows.append(dict(image=img, n_gt=len(gt_items), n_pred=len(preds), recall=np.mean(hits),
                         precision=(np.mean(prec_hits) if preds else 0.0), unsure=res["unsure"],
                         missed=[g for g, h in zip(gt_items, hits) if not h],
                         extra=[pr for pr, h in zip(preds, prec_hits) if not h]))
    tdf = pd.DataFrame(rows)
    tot_gt = tdf["n_gt"].sum()
    item_recall = (tdf["recall"] * tdf["n_gt"]).sum() / tot_gt
    print(f"photos tested   : {len(tdf)}")
    print(f"item recall     : {item_recall:.3f}")
    print(f"mean precision  : {tdf['precision'].mean():.3f}")
    print(f"unsure rate     : {tdf['unsure'].mean():.3f}")
    print(f"photos needing at least one correction: {((tdf['recall'] < 1) | (tdf['precision'] < 1)).mean():.3f}")
    display(tdf.sort_values("recall").head(10))
    tdf.to_csv(RUN_DIR / "team_photo_eval.csv", index=False)
else:
    print(f"No {lab_csv}. Skipping the team-photo test (add photos + labels.csv to run it).")

## 9. Save the model

Writes to `backend/food/model/`:
- `food_yolo.pt` (PyTorch weights)
- `food_yolo.onnx` (what the backend loads with onnxruntime on CPU)
- `classes.txt`, `dataset.yaml`, `model_card.json` (config, metrics, weights used, class list)

Then checks the ONNX output shape and runs one onnxruntime inference to confirm it works. Newer YOLO models can export an **end-to-end (NMS-free)** output, while older ones give a raw `[1, 4+classes, anchors]` output that needs NMS. The check below tells you which one you got so `backend/food/detector.py` can decode it correctly.

In [ ]:
save_dir = Path(CFG["save_dir"]); save_dir.mkdir(parents=True, exist_ok=True)

# 1) PyTorch weights
shutil.copy2(BEST, save_dir / "food_yolo.pt")

# 2) ONNX export
onnx_path = Path(best.export(format="onnx", imgsz=CFG["imgsz"], simplify=True, opset=12, dynamic=False))
shutil.copy2(onnx_path, save_dir / "food_yolo.onnx")

# 3) Class list, dataset yaml, model card
shutil.copy2(Path(CFG["out"]) / "classes.txt", save_dir / "classes.txt")
shutil.copy2(DATA_YAML, save_dir / "dataset.yaml")
card = {
    "model_version": CFG["name"], "weights_used": used_weights, "ultralytics": ultralytics.__version__,
    "imgsz": CFG["imgsz"], "epochs_requested": CFG["epochs"], "train_fraction": CFG["fraction"],
    "n_classes": len(names), "classes": names, "task": CFG["task"],
    "val_mAP50": float(box.map50), "val_mAP50_95": float(box.map),
    "conf_default": CFG["conf"], "unsure_below": CFG["unsure_below"],
    "license_note": "Ultralytics is AGPL-3.0. Credit the dataset in THIRD_PARTY.md.",
}
(save_dir / "model_card.json").write_text(json.dumps(card, indent=2, ensure_ascii=False), encoding="utf-8")
print("saved:", *sorted(p.name for p in save_dir.iterdir()), sep="\n  ")

In [ ]:
# Verify the ONNX file loads and inspect its output tensor (needed by the backend decoder)
import onnxruntime as ort

sess = ort.InferenceSession(str(save_dir / "food_yolo.onnx"), providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
print("input :", inp.name, inp.shape)
for o in sess.get_outputs(): print("output:", o.name, o.shape)

# letterbox preprocessing (same idea the backend uses)
def letterbox(im, size=CFG["imgsz"], color=114):
    W, H = im.size; s = min(size / W, size / H)
    nw, nh = int(round(W * s)), int(round(H * s))
    canvas = Image.new("RGB", (size, size), (color,) * 3)
    px, py = (size - nw) // 2, (size - nh) // 2
    canvas.paste(im.resize((nw, nh), Image.BILINEAR), (px, py))
    return canvas, s, px, py

test_img = Image.open(sample[0]).convert("RGB")
lb, s, px, py = letterbox(test_img)
x = np.asarray(lb, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
t0 = time.time(); out = sess.run(None, {inp.name: x}); dt = (time.time() - t0) * 1000
shape = out[0].shape
print(f"inference: {dt:.0f} ms on CPU | first output shape {shape}")
if len(shape) == 3 and shape[-1] == 6:
    print("-> End-to-end (NMS-free) output: rows are [x1, y1, x2, y2, score, class]. No NMS needed.")
elif len(shape) == 3 and shape[1] == 4 + len(names):
    print("-> Raw output [1, 4+classes, anchors]: decode boxes (cx, cy, w, h) and apply NMS in the backend.")
else:
    print("-> Unrecognized layout. Record this shape in backend/food/README.md and adjust the decoder.")

In [ ]:
# Reload the saved .pt exactly as the backend or a teammate would, and confirm it matches
reloaded = YOLO(str(save_dir / "food_yolo.pt"))
a = detect(best, sample[0]); b = detect(reloaded, sample[0])
print("same top detection after reload:", (a["detections"][:1] == b["detections"][:1]))
print("\nDone. Next: build backend/food/class_to_fndds.json (fuzzy-match classes.txt to FNDDS) and wire POST /food/detect.")